In [0]:
# Databricks notebook source
import dlt
from pyspark.sql.functions import current_timestamp, lit, current_date
from pyspark.sql.types import *

bronze_schema = StructType([
    StructField("material_id", StringType(), True),
    StructField("material_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("uom", StringType(), True),
    StructField("unit_cost", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("plant", StringType(), True),
    StructField("status", StringType(), True),
    StructField("last_updated", StringType(), True),
    StructField("lead_time_days", StringType(), True),
    StructField("safety_stock", StringType(), True),
    StructField("reorder_level", StringType(), True),
    StructField("remarks", StringType(), True)
])


@dlt.table(
    name="bronze_material_master",
    comment="Raw material master data from factories",
    table_properties={"quality": "bronze"}
)
def bronze_material_master():
    """
    Bronze Layer: Read pipe-delimited CSV and store as-is
    """
    raw_df = (spark.read
              .format("csv")
              .option("header", "true")
              .option("delimiter", "|")
              .option("inferSchema", "false")
              .schema(bronze_schema)
              .load("/Volumes/workspace/damg7370/datastore/material_master_1k.csv")
    )
    
    return (raw_df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", lit("/Volumes/workspace/damg7370/datastore/material_master_1k.csv"))
            .withColumn("ingestion_date", current_date())
    )